# Level 2 – Task 1: House Price Prediction with Linear Regression

**Internship:** Oasis Infobyte – Data Analytics  
**Objective:** Predict house values and evaluate a Linear Regression model.

## Workflow
1. Load dataset and inspect data quality.
2. Explore target distribution and correlations.
3. Explain feature selection.
4. Prepare data and split 80/20.
5. Train Linear Regression.
6. Evaluate MSE, RMSE and R².
7. Inspect actual-vs-predicted, residuals and coefficients.
8. Compare with Ridge Regression (bonus).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")


## 1. Load dataset

The California Housing dataset is a public educational dataset. `MedHouseVal` is the target variable (median house value in units of $100,000).

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df.head()


In [ ]:
print("Dataset shape:", df.shape)
display(df.info())
display(df.describe().T)
print("\\nMissing values per column:")
display(df.isna().sum())


## 2. Exploratory Data Analysis

The target distribution helps identify skewness and unusually high values. The correlation matrix gives an initial view of linear relationships; correlation alone does not prove causation.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["MedHouseVal"], bins=40, kde=True)
plt.title("Distribution of Median House Value")
plt.xlabel("Median house value ($100,000s)")
plt.show()

plt.figure(figsize=(11,8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()


## 3. Feature selection discussion

All available predictor columns are used for a baseline model:
- `MedInc`: area median income may capture purchasing power.
- `HouseAge`: housing stock age may relate to value.
- `AveRooms`, `AveBedrms`: average room/bedroom counts describe housing composition.
- `Population`, `AveOccup`: area population and average occupancy describe density.
- `Latitude`, `Longitude`: geographic position captures regional differences.

The target `MedHouseVal` is excluded from the predictors to prevent data leakage.

In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]
numeric_features = X.columns.tolist()

preprocess = ColumnTransformer(
    transformers=[("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features)]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))


## 4. Train Linear Regression and evaluate

MSE penalizes squared errors, RMSE is in the target's units, and R² measures the proportion of variance explained by the model.

In [ ]:
linear_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression())
])
linear_model.fit(X_train, y_train)
pred = linear_model.predict(X_test)

mse = mean_squared_error(y_test, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, pred, alpha=0.25)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual house value")
plt.ylabel("Predicted house value")
plt.title("Actual vs Predicted Prices")
plt.show()

residuals = y_test - pred
plt.figure(figsize=(7,5))
plt.scatter(pred, residuals, alpha=0.25)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted house value")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual Plot")
plt.show()


## 5. Coefficient analysis

Coefficients are expressed on standardized feature scales because the pipeline standardizes predictors. Positive coefficients increase the prediction while negative coefficients decrease it, holding other features constant.

In [ ]:
coef = linear_model.named_steps["model"].coef_
coef_df = pd.DataFrame({"Feature": numeric_features, "Coefficient": coef})
coef_df = coef_df.reindex(coef_df["Coefficient"].abs().sort_values(ascending=False).index)
display(coef_df)

plt.figure(figsize=(9,5))
sns.barplot(data=coef_df, x="Coefficient", y="Feature")
plt.title("Linear Regression Coefficients (standardized features)")
plt.show()


## 6. Bonus: Compare Ridge Regression

Ridge adds L2 regularization, which can reduce coefficient magnitude. Compare its test-set metrics with the baseline.

In [ ]:
ridge_model = Pipeline([
    ("preprocess", preprocess),
    ("model", Ridge(alpha=1.0))
])
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)

comparison = pd.DataFrame([
    {"Model":"Linear Regression",
     "MSE":mean_squared_error(y_test, pred),
     "RMSE":np.sqrt(mean_squared_error(y_test, pred)),
     "R2":r2_score(y_test, pred)},
    {"Model":"Ridge Regression",
     "MSE":mean_squared_error(y_test, ridge_pred),
     "RMSE":np.sqrt(mean_squared_error(y_test, ridge_pred)),
     "R2":r2_score(y_test, ridge_pred)}
])
display(comparison)


## 7. Conclusion

Summarize the observed test metrics, the strongest positive/negative coefficients, and any patterns in the residual plot. Mention that a simple linear model may not capture all geographic and nonlinear effects. This model is for educational analysis, not real-world valuation.